In [41]:
from dotenv import load_dotenv
load_dotenv()

True

In [42]:
from google import genai
gemini_client = genai.Client() # picks up the API key from the env variable GEMINI_API_KEY

In [43]:
def llm(prompt):
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

In [44]:
question = 'I have just discovered the course, can I join it now?'
answer = llm(question)
print(answer)

I’d love to help you check, but I need a little more context! 

Could you tell me:
1. What is the **name of the course**?
2. Which **platform, school, or organization** is offering it (e.g., Coursera, Udemy, a specific university, a local workshop)?

***

**In general, here is how it usually works:**

* **Self-paced online courses** (like Udemy, Coursera, or edX): You can almost always join and start **immediately**, anytime you want.
* **Live or cohort-based courses** (like university classes or bootcamps): It depends on whether registration is still open or if they allow late enrollment.

If you reply with the course details or a link, I can give you a specific answer!


In [45]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [46]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [47]:
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can also just start learning and submitting homework (while the form is open) without registering.


In [48]:

def rag(question):
    """
    Retrieval-Augmented Generation
    1. Retrieve relevant search results
    2. Build a prompt with the retrieved documents and the question
    3. Generate an answer using the LLM
    """
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [49]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [50]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 144},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253}]

In [51]:
documents = [] # list of questions from all courses
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url) # fetch json file of all questions in the course, from its url
    course_response.raise_for_status() # raise an error if the request was unsuccessful
    course_data = course_response.json() # parse the json response into a Python object (list of questions)

    documents.extend(course_data)

len(documents)

1406

In [52]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"], # fields for searching
    keyword_fields = ["course"] # existing fields in the dataset for filtering
)

index.fit(documents)

In [53]:
def search(question, course='llm-zoomcamp'):
    """
    Search for relevant documents based on the question and the specified course.
    """
    boost_dict = {"question": 2.0, "section": 0.5} # importance of fields for scoring
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5
    )

In [54]:
search_results = search(question)

In [55]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [56]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [57]:
def build_context(search_results):
    """ 
    Build a context dictionary from the search results to be used in the prompt for the LLM.
    """
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [58]:
context = build_context(search_results)
print(context)

USER_PROMPT_TEMPLATE.format(question=question, context=context) # setting the question and context in the prompt template for the LLM

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to fin

'\nQuestion:\nI have just discovered the course, can I join it now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a ce

In [59]:
def build_prompt(question, search_results):
    """
    Build a prompt for the LLM using the question and the search results.
    """
    context = build_context(search_results)
    
    # setting the question and context in the prompt template for the LLM
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip() # remove \n at the beginning and end of the prompt

In [60]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I have just discovered the course, can I join it now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish

In [61]:
response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
)

response.text
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=48,
  prompt_token_count=521,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=521
    ),
  ],
  thoughts_token_count=333,
  total_token_count=902
)

In [62]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage_metadata.prompt_token_count * input_price +
    response.usage_metadata.candidates_token_count * output_price
)

cost

0.00060675

In [63]:
from google.genai import types

# Passing user's prompt seperately
conversation_history = [
    {'role': 'user', 'parts': [{'text': prompt}]}, # Varies
]

response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=conversation_history,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS, # Passing INSTRUCTIONS to the system seperately
        )
)

print(response.text)
response.usage_metadata

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while project submissions are still being accepted.


GenerateContentResponseUsageMetadata(
  candidates_token_count=33,
  prompt_token_count=575,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=575
    ),
  ],
  thoughts_token_count=223,
  total_token_count=831
)

In [64]:
def llm(instructions, user_prompt, model="gemini-3.6-flash"):

    conversation_history = [
    {'role': 'user', 'parts': [{'text': prompt}]},
    ]

    response = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=conversation_history,
            config=types.GenerateContentConfig(
                system_instruction=INSTRUCTIONS,
            )
    )

    return response.text

In [65]:
def rag(query, model="gemini-3.6-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [66]:
answer = rag(question)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
